# ДЗ 1.3. 

## Подключение

In [1]:
%pip install -q clickhouse-connect

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import pandas as pd
import clickhouse_connect

STUDENT_N = 43
DB = f"student{STUDENT_N}"
CL = "homework_cluster"

client = clickhouse_connect.get_client(
    host=os.environ.get("CH_HOST", "localhost"),
    port=int(os.environ.get("CH_PORT", "8123")),
    username="default",
    password=os.environ.get("CH_PASSWORD", "homework"),
)


def S(sql):
    return sql.replace("$DB", DB).replace("$CL", CL)


def run(sql, **settings):
    client.command(S(sql), settings=settings or None)
    print("OK:", " ".join(S(sql).split())[:90])


def q(sql, **settings):
    return client.query_df(S(sql), settings=settings or None)


print(client.command("SELECT version()"))

25.8.33.6


In [3]:
display(q("SELECT cluster, shard_num, replica_num, host_name FROM system.clusters WHERE cluster = '$CL'"))
display(q("SELECT hostName() AS host, macro, substitution FROM clusterAllReplicas('$CL', system.macros) ORDER BY host, macro"))

,cluster,shard_num,replica_num,host_name
0,homework_cluster,1,1,ch11
1,homework_cluster,1,2,ch12
2,homework_cluster,2,1,ch21
3,homework_cluster,2,2,ch22


,host,macro,substitution
0,ch11,replica,ch11
1,ch11,shard,01
2,ch12,replica,ch12
3,ch12,shard,01
4,ch21,replica,ch21
5,ch21,shard,02
6,ch22,replica,ch22
7,ch22,shard,02


## Исходные данные

In [4]:
if str(client.command("EXISTS TABLE homework_data.push_events_source")).strip() != "1":
    for st in open("generator.sql", encoding="utf-8").read().split(";"):
        if st.strip():
            run(st)

q('''
SELECT count() AS rows, uniqExact(event_id) AS unique_event_ids, uniqExact(user_id) AS users,
       min(event_dt) AS min_dt, max(event_dt) AS max_dt
FROM homework_data.push_events_source
''')

,rows,unique_event_ids,users,min_dt,max_dt
0,2100658,2000000,10000,2026-09-14 19:10:35,2026-10-14 21:08:52


In [5]:
run("DROP DATABASE IF EXISTS $DB ON CLUSTER $CL SYNC")
run("CREATE DATABASE $DB ON CLUSTER $CL")

OK: DROP DATABASE IF EXISTS student43 ON CLUSTER homework_cluster SYNC
OK: CREATE DATABASE student43 ON CLUSTER homework_cluster


## Задание 1. 

In [6]:
run('''
CREATE TABLE $DB.push_events_raw_local ON CLUSTER $CL
(
    event_dt   DateTime,
    event_id   UInt64,
    user_id    UInt64,
    message_id UInt64,
    platform   LowCardinality(String),
    status     LowCardinality(String),
    latency_ms UInt32
)
ENGINE = ReplicatedMergeTree('/clickhouse/tables/{shard}/{database}/{table}', '{replica}')
PARTITION BY toYYYYMMDD(event_dt)
ORDER BY (user_id, event_dt)
TTL event_dt + INTERVAL 30 DAY
''')

run('''
CREATE TABLE $DB.push_events_raw ON CLUSTER $CL AS $DB.push_events_raw_local
ENGINE = Distributed('$CL', '$DB', 'push_events_raw_local', cityHash64(user_id))
''')

OK: CREATE TABLE student43.push_events_raw_local ON CLUSTER homework_cluster ( event_dt DateTi
OK: CREATE TABLE student43.push_events_raw ON CLUSTER homework_cluster AS student43.push_event


Данные загружаются после создания всех MV (задания 2 и 3), иначе MV их не увидят.

## Задание 2. 

In [7]:
run('''
CREATE TABLE $DB.push_events_agg_local ON CLUSTER $CL
(
    event_date  Date,
    platform    LowCardinality(String),
    events_cnt  AggregateFunction(count),
    users_uniq  AggregateFunction(uniq, UInt64),
    latency_avg AggregateFunction(avg, UInt32)
)
ENGINE = ReplicatedAggregatingMergeTree('/clickhouse/tables/{shard}/{database}/{table}', '{replica}')
PARTITION BY toYYYYMM(event_date)
ORDER BY (event_date, platform)
''')

run('''
CREATE TABLE $DB.push_events_agg ON CLUSTER $CL AS $DB.push_events_agg_local
ENGINE = Distributed('$CL', '$DB', 'push_events_agg_local', rand())
''')

run('''
CREATE MATERIALIZED VIEW $DB.push_events_agg_mv ON CLUSTER $CL
TO $DB.push_events_agg_local
AS SELECT
    toDate(event_dt)     AS event_date,
    platform,
    countState()         AS events_cnt,
    uniqState(user_id)   AS users_uniq,
    avgState(latency_ms) AS latency_avg
FROM $DB.push_events_raw_local
GROUP BY event_date, platform
''')

OK: CREATE TABLE student43.push_events_agg_local ON CLUSTER homework_cluster ( event_date Date
OK: CREATE TABLE student43.push_events_agg ON CLUSTER homework_cluster AS student43.push_event
OK: CREATE MATERIALIZED VIEW student43.push_events_agg_mv ON CLUSTER homework_cluster TO stude


## Задание 3.

In [8]:
run('''
CREATE TABLE $DB.push_counters_local ON CLUSTER $CL
(
    event_date      Date,
    platform        LowCardinality(String),
    user_id         UInt64,
    delivered_count UInt64,
    failed_count    UInt64,
    opened_count    UInt64
)
ENGINE = ReplicatedSummingMergeTree('/clickhouse/tables/{shard}/{database}/{table}', '{replica}')
PARTITION BY toYYYYMM(event_date)
ORDER BY (event_date, platform)
''')

run('''
CREATE TABLE $DB.push_counters ON CLUSTER $CL AS $DB.push_counters_local
ENGINE = Distributed('$CL', '$DB', 'push_counters_local', rand())
''')

run('''
CREATE MATERIALIZED VIEW $DB.push_counters_mv ON CLUSTER $CL
TO $DB.push_counters_local
AS SELECT
    toDate(event_dt)              AS event_date,
    platform,
    any(user_id)                  AS user_id,
    countIf(status = 'delivered') AS delivered_count,
    countIf(status = 'failed')    AS failed_count,
    countIf(status = 'opened')    AS opened_count
FROM $DB.push_events_raw_local
GROUP BY event_date, platform
''')

OK: CREATE TABLE student43.push_counters_local ON CLUSTER homework_cluster ( event_date Date, 
OK: CREATE TABLE student43.push_counters ON CLUSTER homework_cluster AS student43.push_counter
OK: CREATE MATERIALIZED VIEW student43.push_counters_mv ON CLUSTER homework_cluster TO student


## Загрузка данных в raw через Distributed

In [9]:
run('''
INSERT INTO $DB.push_events_raw (event_dt, event_id, user_id, message_id, platform, status, latency_ms)
SELECT event_dt, event_id, user_id, message_id, platform, status, latency_ms
FROM homework_data.push_events_source
''')

q('''
SELECT (SELECT count() FROM homework_data.push_events_source) AS source_rows,
       (SELECT count() FROM $DB.push_events_raw) AS raw_rows,
       (SELECT countIf(event_dt < now() - INTERVAL 30 DAY) FROM homework_data.push_events_source) AS older_than_30d
''')

OK: INSERT INTO student43.push_events_raw (event_dt, event_id, user_id, message_id, platform, 


,source_rows,raw_rows,older_than_30d
0,2100658,2098649,2009


## Задание 1. Проверки

In [10]:
q("SELECT hostName() AS host, count() AS rows FROM $DB.push_events_raw GROUP BY host ORDER BY host")

,host,rows
0,ch11,1061694
1,ch22,1036955


In [11]:
q('''
SELECT hostName() AS host, count() AS rows
FROM clusterAllReplicas('$CL', '$DB', 'push_events_raw_local')
GROUP BY host ORDER BY host
''')

,host,rows
0,ch11,1061694
1,ch12,1061694
2,ch21,1036955
3,ch22,1036955


In [12]:
q('''
SELECT hostName() AS host, database, table, is_readonly, absolute_delay, log_pointer
FROM clusterAllReplicas('$CL', system.replicas)
WHERE database = '$DB'
ORDER BY table, host
''')

,host,database,table,is_readonly,absolute_delay,log_pointer
0,ch11,student43,push_counters_local,0,0,2
1,ch12,student43,push_counters_local,0,0,2
2,ch21,student43,push_counters_local,0,0,2
3,ch22,student43,push_counters_local,0,0,2
4,ch11,student43,push_events_agg_local,0,0,2
5,ch12,student43,push_events_agg_local,0,0,2
6,ch21,student43,push_events_agg_local,0,0,2
7,ch22,student43,push_events_agg_local,0,0,2
8,ch11,student43,push_events_raw_local,0,0,33
9,ch12,student43,push_events_raw_local,0,0,33


Вопрос: что будет, если вставлять в локальную таблицу, а не в Distributed?

Все строки попадут на один шард, к которому подключён клиент и sharding key не применяется, второй шард останется пустым, а репликация и MV сработают только на этом шарде. Дедупликация по event_id перестанет работать. Чтение через Distributed при этом покажет все данные.

## Задание 2. Проверки

Сравниваем за последние 28 дней.

In [13]:
CMP_FROM = "today() - 27"

agg_df = q(f'''
SELECT event_date, platform,
       countMerge(events_cnt) AS events,
       uniqMerge(users_uniq) AS users,
       round(avgMerge(latency_avg), 4) AS avg_latency
FROM $DB.push_events_agg
WHERE event_date >= {CMP_FROM}
GROUP BY event_date, platform
ORDER BY event_date, platform
''')
raw_df = q(f'''
SELECT toDate(event_dt) AS event_date, platform,
       count() AS events,
       uniq(user_id) AS users,
       round(avg(latency_ms), 4) AS avg_latency
FROM $DB.push_events_raw
WHERE event_dt >= toDateTime({CMP_FROM})
GROUP BY event_date, platform
ORDER BY event_date, platform
''')
print("agg == raw:", agg_df.equals(raw_df))
agg_df.head(8)

agg == raw: True


,event_date,platform,events,users,avg_latency
0,2026-09-17,ANDROID,17699,8116,2503.5371
1,2026-09-17,DESKTOP,17490,8137,2492.6079
2,2026-09-17,IOS,17576,8099,2513.3500
3,2026-09-17,WEB,17668,8160,2513.3370
4,2026-09-18,ANDROID,17492,8166,2516.5560
5,2026-09-18,DESKTOP,17448,8127,2530.6014
6,2026-09-18,IOS,17358,8127,2503.9117
7,2026-09-18,WEB,17249,8096,2495.6409


In [14]:
ROWS_PER_NODE = '''
SELECT hostName() AS host, count() AS physical_rows, uniqExact(event_date, platform) AS distinct_keys
FROM clusterAllReplicas('$CL', '$DB', 'push_events_agg_local')
GROUP BY host ORDER BY host
'''
display(q(ROWS_PER_NODE))

run("OPTIMIZE TABLE $DB.push_events_agg_local ON CLUSTER $CL FINAL")
run("SYSTEM SYNC REPLICA ON CLUSTER $CL $DB.push_events_agg_local")
display(q(ROWS_PER_NODE))

,host,physical_rows,distinct_keys
0,ch11,124,124
1,ch12,124,124
2,ch21,124,124
3,ch22,124,124


OK: OPTIMIZE TABLE student43.push_events_agg_local ON CLUSTER homework_cluster FINAL
OK: SYSTEM SYNC REPLICA ON CLUSTER homework_cluster student43.push_events_agg_local


,host,physical_rows,distinct_keys
0,ch11,124,124
1,ch12,124,124
2,ch21,124,124
3,ch22,124,124


Проблема -  MV, созданный после вставки.

In [15]:
run('''
CREATE TABLE $DB.push_events_agg_late_local ON CLUSTER $CL AS $DB.push_events_agg_local
ENGINE = ReplicatedAggregatingMergeTree('/clickhouse/tables/{shard}/{database}/{table}', '{replica}')
PARTITION BY toYYYYMM(event_date)
ORDER BY (event_date, platform)
''')
run('''
CREATE TABLE $DB.push_events_agg_late ON CLUSTER $CL AS $DB.push_events_agg_late_local
ENGINE = Distributed('$CL', '$DB', 'push_events_agg_late_local', rand())
''')
run('''
CREATE MATERIALIZED VIEW $DB.push_events_agg_late_mv ON CLUSTER $CL
TO $DB.push_events_agg_late_local
AS SELECT toDate(event_dt) AS event_date, platform,
          countState() AS events_cnt, uniqState(user_id) AS users_uniq, avgState(latency_ms) AS latency_avg
FROM $DB.push_events_raw_local
GROUP BY event_date, platform
''')
q("SELECT count() AS rows FROM $DB.push_events_agg_late")

OK: CREATE TABLE student43.push_events_agg_late_local ON CLUSTER homework_cluster AS student43
OK: CREATE TABLE student43.push_events_agg_late ON CLUSTER homework_cluster AS student43.push_
OK: CREATE MATERIALIZED VIEW student43.push_events_agg_late_mv ON CLUSTER homework_cluster TO 


,rows
0,0


Вопрос: что произойдёт, если MV создать после вставки данных в raw?

Таблица останется пустой, так как MV срабатывает только на новые INSERT и не перечитывает уже имеющиеся данные, чтобы исправить, нужно заполнить таблицу тем же запросом через INSERT SELECT .

In [16]:
run('''
INSERT INTO $DB.push_events_agg_late
SELECT toDate(event_dt) AS event_date, platform,
       countState(), uniqState(user_id), avgState(latency_ms)
FROM $DB.push_events_raw
GROUP BY event_date, platform
''')
late_df = q(f'''
SELECT event_date, platform, countMerge(events_cnt) AS events, uniqMerge(users_uniq) AS users,
       round(avgMerge(latency_avg), 4) AS avg_latency
FROM $DB.push_events_agg_late
WHERE event_date >= {CMP_FROM}
GROUP BY event_date, platform
ORDER BY event_date, platform
''')
print("после backfill late == raw:", late_df.equals(raw_df))

OK: INSERT INTO student43.push_events_agg_late SELECT toDate(event_dt) AS event_date, platform
после backfill late == raw: True


Замечание,  avgMerge без GROUP BY.

In [17]:
print("avgMerge без GROUP BY:", client.command(S(f"SELECT round(avgMerge(latency_avg), 4) FROM $DB.push_events_agg WHERE event_date >= {CMP_FROM}")))
print("avg по raw:           ", client.command(S(f"SELECT round(avg(latency_ms), 4) FROM $DB.push_events_raw WHERE event_dt >= toDateTime({CMP_FROM})")))
print("среднее средних:      ", client.command(S(f"SELECT round(avg(finalizeAggregation(latency_avg)), 4) FROM $DB.push_events_agg WHERE event_date >= {CMP_FROM}")))
try:
    client.command(S("SELECT event_date, platform, avgMerge(latency_avg) FROM $DB.push_events_agg"))
except Exception as e:
    print("с ключами без GROUP BY:", str(e).splitlines()[0][:120])

avgMerge без GROUP BY: 2501.6303
avg по raw:            2501.6303
среднее средних:       2501.5961
с ключами без GROUP BY: Received ClickHouse exception, code: 215, server response: Code: 215. DB::Exception: Column 'student43.push_events_agg.e


Вопрос: что вернёт avgMerge без GROUP BY? Будет ли это корректным средним?

Вернёт одно общее среднее по всем строкам, и оно будет корректное, потому что состояние avg хранит сумму и количество, и слияние даёт взвешенное среднее. Средних по дням и платформам при этом нет, с ключевыми колонками без GROUP BY запрос уходит в ошибку. Нельзя усреднять уже посчитанные средние (finalizeAggregation + avg), это среднее средних без учёта числа событий.

## Задание 3. Проверки

In [18]:
COUNTERS = "SELECT sum(delivered_count) AS delivered, sum(failed_count) AS failed, sum(opened_count) AS opened FROM $DB.push_counters"
before = q(COUNTERS)

run('''
INSERT INTO $DB.push_events_raw (event_dt, event_id, user_id, message_id, platform, status, latency_ms)
SELECT now() - number % 3600, 100000000 + number, number % 10000, number,
       ['ANDROID', 'IOS', 'WEB', 'DESKTOP'][number % 4 + 1],
       ['delivered', 'failed', 'opened'][number % 3 + 1],
       toUInt32(1 + number % 5000)
FROM numbers(3000)
''')

after = q(COUNTERS)
print("прирост:", (after - before).iloc[0].to_dict())

OK: INSERT INTO student43.push_events_raw (event_dt, event_id, user_id, message_id, platform, 
прирост: {'delivered': 1000, 'failed': 1000, 'opened': 1000}


In [19]:
cnt_df = q(f'''
SELECT event_date, platform, sum(delivered_count) AS delivered, sum(failed_count) AS failed, sum(opened_count) AS opened
FROM $DB.push_counters WHERE event_date >= {CMP_FROM}
GROUP BY event_date, platform ORDER BY event_date, platform
''')
raw_cnt_df = q(f'''
SELECT toDate(event_dt) AS event_date, platform,
       countIf(status = 'delivered') AS delivered, countIf(status = 'failed') AS failed, countIf(status = 'opened') AS opened
FROM $DB.push_events_raw WHERE event_dt >= toDateTime({CMP_FROM})
GROUP BY event_date, platform ORDER BY event_date, platform
''')
print("counters == raw:", cnt_df.equals(raw_cnt_df))

counters == raw: True


In [20]:
q("SELECT * FROM $DB.push_counters ORDER BY event_date DESC, platform LIMIT 8")

,event_date,platform,user_id,delivered_count,failed_count,opened_count
0,2026-10-14,ANDROID,0,134,118,126
1,2026-10-14,ANDROID,10758,2456,2614,2467
2,2026-10-14,ANDROID,16,116,132,124
3,2026-10-14,ANDROID,86,2363,2481,2462
4,2026-10-14,DESKTOP,3,120,146,122
5,2026-10-14,DESKTOP,3822,2437,2442,2555
6,2026-10-14,DESKTOP,15,130,104,128
7,2026-10-14,DESKTOP,977,2435,2389,2505


In [21]:
run("OPTIMIZE TABLE $DB.push_counters_local ON CLUSTER $CL FINAL")
run("SYSTEM SYNC REPLICA ON CLUSTER $CL $DB.push_counters_local")
display(q("SELECT * FROM $DB.push_counters ORDER BY event_date DESC, platform LIMIT 8"))
q("SELECT max(user_id) AS max_user_id_in_counters, (SELECT max(user_id) FROM $DB.push_events_raw) AS max_real_user_id FROM $DB.push_counters")

OK: OPTIMIZE TABLE student43.push_counters_local ON CLUSTER homework_cluster FINAL
OK: SYSTEM SYNC REPLICA ON CLUSTER homework_cluster student43.push_counters_local


,event_date,platform,user_id,delivered_count,failed_count,opened_count
0,2026-10-14,ANDROID,10758,2590,2732,2593
1,2026-10-14,ANDROID,102,2479,2613,2586
2,2026-10-14,DESKTOP,3825,2557,2588,2677
3,2026-10-14,DESKTOP,992,2565,2493,2633
4,2026-10-14,IOS,1341,2528,2642,2674
5,2026-10-14,IOS,5661,2508,2620,2505
6,2026-10-14,WEB,1605,2623,2620,2579
7,2026-10-14,WEB,5566,2671,2473,2532


,max_user_id_in_counters,max_real_user_id
0,10758,9999


Вопрос: что не так с user_id после OPTIMIZE FINAL? Почему? Как починить?

user_id стал суммой id из слитых строк, при этом таких значений в данных нет. SummingMergeTree при слиянии строк с одинаковым ключом сортировки суммирует все числовые колонки вне ключа, и также user_id тоже суммируется.

Можно исправить, если 
1. Убрать user_id из таблицы или явно перечислить суммируемые колонки.
2. Добавить user_id в ORDER BY и в GROUP BY в MV, тогда это будут счётчики по пользователю.
Ниже вариант 1.

In [22]:
run('''
CREATE TABLE $DB.push_counters_fixed_local ON CLUSTER $CL
(
    event_date      Date,
    platform        LowCardinality(String),
    delivered_count UInt64,
    failed_count    UInt64,
    opened_count    UInt64
)
ENGINE = ReplicatedSummingMergeTree('/clickhouse/tables/{shard}/{database}/{table}', '{replica}',
                                    (delivered_count, failed_count, opened_count))
PARTITION BY toYYYYMM(event_date)
ORDER BY (event_date, platform)
''')
run('''
CREATE TABLE $DB.push_counters_fixed ON CLUSTER $CL AS $DB.push_counters_fixed_local
ENGINE = Distributed('$CL', '$DB', 'push_counters_fixed_local', rand())
''')
run('''
CREATE MATERIALIZED VIEW $DB.push_counters_fixed_mv ON CLUSTER $CL
TO $DB.push_counters_fixed_local
AS SELECT toDate(event_dt) AS event_date, platform,
          countIf(status = 'delivered') AS delivered_count,
          countIf(status = 'failed')    AS failed_count,
          countIf(status = 'opened')    AS opened_count
FROM $DB.push_events_raw_local
GROUP BY event_date, platform
''')
run('''
INSERT INTO $DB.push_counters_fixed
SELECT toDate(event_dt) AS event_date, platform,
       countIf(status = 'delivered'), countIf(status = 'failed'), countIf(status = 'opened')
FROM $DB.push_events_raw
GROUP BY event_date, platform
''')
run("OPTIMIZE TABLE $DB.push_counters_fixed_local ON CLUSTER $CL FINAL")
run("SYSTEM SYNC REPLICA ON CLUSTER $CL $DB.push_counters_fixed_local")

fixed_df = q(f'''
SELECT event_date, platform, sum(delivered_count) AS delivered, sum(failed_count) AS failed, sum(opened_count) AS opened
FROM $DB.push_counters_fixed WHERE event_date >= {CMP_FROM}
GROUP BY event_date, platform ORDER BY event_date, platform
''')
print("fixed == raw:", fixed_df.equals(raw_cnt_df))
q("SELECT * FROM $DB.push_counters_fixed ORDER BY event_date DESC, platform LIMIT 5")

OK: CREATE TABLE student43.push_counters_fixed_local ON CLUSTER homework_cluster ( event_date 
OK: CREATE TABLE student43.push_counters_fixed ON CLUSTER homework_cluster AS student43.push_c
OK: CREATE MATERIALIZED VIEW student43.push_counters_fixed_mv ON CLUSTER homework_cluster TO s
OK: INSERT INTO student43.push_counters_fixed SELECT toDate(event_dt) AS event_date, platform,
OK: OPTIMIZE TABLE student43.push_counters_fixed_local ON CLUSTER homework_cluster FINAL
OK: SYSTEM SYNC REPLICA ON CLUSTER homework_cluster student43.push_counters_fixed_local
fixed == raw: True


,event_date,platform,delivered_count,failed_count,opened_count
0,2026-10-14,ANDROID,2590,2732,2593
1,2026-10-14,ANDROID,2479,2613,2586
2,2026-10-14,DESKTOP,2557,2588,2677
3,2026-10-14,DESKTOP,2565,2493,2633
4,2026-10-14,IOS,2528,2642,2674


Вопрос: почему SummingMergeTree не подходит для расчёта avg?

Он только складывает, а среднее не аддитивно, так сумма средних не даёт среднего, поэтому нужно хранить сумму и количество и делить при чтении, либо использовать AggregatingMergeTree с avgState.

## Задание 4. Дедупликация


In [23]:
q('''
SELECT count() AS duplicate_ids, countIf(toDate(min_dt) != toDate(max_dt)) AS cross_day_pairs
FROM (
    SELECT event_id, min(event_dt) AS min_dt, max(event_dt) AS max_dt
    FROM homework_data.push_events_source
    GROUP BY event_id
    HAVING count() > 1
)
''')

,duplicate_ids,cross_day_pairs
0,100658,4147


In [24]:
run('''
CREATE TABLE $DB.push_events_dedup_local ON CLUSTER $CL
(
    event_dt   DateTime,
    event_id   UInt64,
    user_id    UInt64,
    message_id UInt64,
    platform   LowCardinality(String),
    status     LowCardinality(String),
    latency_ms UInt32
)
ENGINE = ReplicatedReplacingMergeTree('/clickhouse/tables/{shard}/{database}/{table}', '{replica}', event_dt)
ORDER BY event_id
''')
run('''
CREATE TABLE $DB.push_events_dedup ON CLUSTER $CL AS $DB.push_events_dedup_local
ENGINE = Distributed('$CL', '$DB', 'push_events_dedup_local', cityHash64(event_id))
''')
run('''
INSERT INTO $DB.push_events_dedup (event_dt, event_id, user_id, message_id, platform, status, latency_ms)
SELECT event_dt, event_id, user_id, message_id, platform, status, latency_ms
FROM homework_data.push_events_source
''')

OK: CREATE TABLE student43.push_events_dedup_local ON CLUSTER homework_cluster ( event_dt Date
OK: CREATE TABLE student43.push_events_dedup ON CLUSTER homework_cluster AS student43.push_eve
OK: INSERT INTO student43.push_events_dedup (event_dt, event_id, user_id, message_id, platform


In [25]:
DEDUP_CHECK = '''
SELECT (SELECT count() FROM $DB.push_events_dedup) AS count_plain,
       (SELECT count() FROM $DB.push_events_dedup FINAL) AS count_final,
       (SELECT uniqExact(event_id) FROM homework_data.push_events_source) AS unique_event_ids
'''
q(DEDUP_CHECK)

,count_plain,count_final,unique_event_ids
0,2000000,2000000,2000000


In [26]:
run("OPTIMIZE TABLE $DB.push_events_dedup_local ON CLUSTER $CL FINAL")
run("SYSTEM SYNC REPLICA ON CLUSTER $CL $DB.push_events_dedup_local")
q(DEDUP_CHECK)

OK: OPTIMIZE TABLE student43.push_events_dedup_local ON CLUSTER homework_cluster FINAL
OK: SYSTEM SYNC REPLICA ON CLUSTER homework_cluster student43.push_events_dedup_local


,count_plain,count_final,unique_event_ids
0,2000000,2000000,2000000


In [27]:
q('''
SELECT countIf(d.event_dt = s.max_dt) AS kept_max_version,
       countIf(d.event_dt = s.min_dt) AS kept_min_version,
       count() AS duplicate_ids
FROM (SELECT event_id, event_dt FROM $DB.push_events_dedup FINAL) AS d
INNER JOIN (
    SELECT event_id, min(event_dt) AS min_dt, max(event_dt) AS max_dt
    FROM homework_data.push_events_source
    GROUP BY event_id
    HAVING count() > 1
) AS s USING (event_id)
''')

,kept_max_version,kept_min_version,duplicate_ids
0,100658,0,100658


Вопрос: почему count() и count() FINAL разные? Когда дубли реально удаляются?

Дубли удаляются только при слиянии частей, то есть в неизвестный момент, внутри одной партиции и одного шарда. Пока дубль и оригинал в разных частях, count() видит обе строки. FINAL схлопывает строки при чтении, поэтому count() FINAL равен числу уникальных event_id. После OPTIMIZE FINAL дубли удалены, и значения совпадают.

Вопрос: какая версия выигрывает при мёрдже?

Максимальная, и при этом остаётся строка с наибольшим event_dt. При равных версиях остаётся последняя вставленная.

Вопрос: можно ли полагаться на ReplacingMergeTree для строгой дедупликации в моменте?

Нет, дедупликация при этом отложенная. Для точного результата нужен FINAL при чтении или явная дедупликация в запросе, например argMax(..., event_dt) с GROUP BY event_id или LIMIT 1 BY event_id.

## Задание 5. Сравнение производительности

In [28]:
RAW_Q = '''
SELECT platform, count() AS events, uniq(user_id) AS users, round(avg(latency_ms), 4) AS avg_latency
FROM $DB.push_events_raw
WHERE event_dt >= toDateTime(today() - 6)
GROUP BY platform
ORDER BY platform
'''
AGG_Q = '''
SELECT platform, countMerge(events_cnt) AS events, uniqMerge(users_uniq) AS users,
       round(avgMerge(latency_avg), 4) AS avg_latency
FROM $DB.push_events_agg
WHERE event_date >= today() - 6
GROUP BY platform
ORDER BY platform
'''
for _ in range(3):
    r_raw = q(RAW_Q, log_comment="hw13_raw")
    r_agg = q(AGG_Q, log_comment="hw13_agg")

print("результаты совпадают:", r_raw.equals(r_agg))
r_raw

результаты совпадают: True


,platform,events,users,avg_latency
0,ANDROID,121217,10000,2491.4174
1,DESKTOP,120503,10000,2493.5293
2,IOS,119941,10000,2497.1179
3,WEB,120978,10000,2491.4941


In [29]:
run("SYSTEM FLUSH LOGS")
perf = q('''
SELECT log_comment AS variant, event_time, query_duration_ms, read_rows, formatReadableSize(read_bytes) AS read_size
FROM system.query_log
WHERE event_time > now() - 120
  AND type = 'QueryFinish'
  AND query LIKE '%push_events%'
  AND log_comment IN ('hw13_raw', 'hw13_agg')
  AND is_initial_query
ORDER BY event_time DESC
LIMIT 10
''')
display(perf)
perf.groupby("variant")[["query_duration_ms", "read_rows"]].median()

OK: SYSTEM FLUSH LOGS


,variant,event_time,query_duration_ms,read_rows,read_size
0,hw13_raw,2026-10-14 20:50:23,18,482639,7.83 MiB
1,hw13_agg,2026-10-14 20:50:23,18,120,15.59 KiB
2,hw13_raw,2026-10-14 20:50:23,16,482639,7.83 MiB
3,hw13_agg,2026-10-14 20:50:23,18,120,15.59 KiB
4,hw13_raw,2026-10-14 20:50:23,19,482639,7.83 MiB
5,hw13_agg,2026-10-14 20:50:23,20,120,15.59 KiB


,query_duration_ms,read_rows
variant,,
hw13_agg,18.0,120.0
hw13_raw,18.0,482639.0


Вопрос: в каком случае raw может быть быстрее agg?

Когда агрегат почти не сокращает данные и слияние состояний uniq дороже обычного подсчёта, или когда запрос попадает в первичный ключ raw. Также на очень маленьких объёмах, где накладные расходы больше самого чтения.

## Итог

In [30]:
print("База:", DB)
display(q("SELECT hostName() AS host, count() AS rows FROM $DB.push_events_raw GROUP BY host ORDER BY host"))
display(q("SELECT hostName() AS host, table, is_readonly, absolute_delay FROM clusterAllReplicas('$CL', system.replicas) WHERE database = '$DB' ORDER BY table, host"))
perf.groupby("variant")[["query_duration_ms", "read_rows"]].median()

База: student43


,host,rows
0,ch11,1063194
1,ch21,1038453


,host,table,is_readonly,absolute_delay
0,ch11,push_counters_fixed_local,0,0
1,ch12,push_counters_fixed_local,0,0
2,ch21,push_counters_fixed_local,0,0
3,ch22,push_counters_fixed_local,0,0
4,ch11,push_counters_local,0,0
5,ch12,push_counters_local,0,0
6,ch21,push_counters_local,0,0
7,ch22,push_counters_local,0,0
8,ch11,push_events_agg_late_local,0,0
9,ch12,push_events_agg_late_local,0,0


,query_duration_ms,read_rows
variant,,
hw13_agg,18.0,120.0
hw13_raw,18.0,482639.0
